In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import pandas as pd
import numpy as np
import torch
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Fixed Income Library is loaded.


### Create Build Method Collection (USD)

Curves:
- `SOFR-1B` -- root overnight index curve
- `USD-Federal Funds-H.15-1B` -- overnight index curve, references `SOFR-1B`
- `USD-LIBOR-BBA-3M` -- LIBOR-style index curve, references `SOFR-1B`
- `SOFR-1B-FLAT` / `USD-Federal Funds-H.15-1B-FLAT` -- flat (zero-spread) funding curves over their respective indices
- `USD` -- common build method tying the currency to its CSA/funding curve

All components are calibrated directly from `INSTANTANEOUS FORWARD RATE` state data, so this exercises the `create_yield_curve_from_state_data` path in `YieldCurveBuilder`.

In [2]:
bm_list = []

bm_list.append(qfCreateBuildMethod('YC_OVERNIGHT_INDEX_ELEMENT', {
    'TARGET': 'SOFR-1B',
    'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-IFR',
}))

bm_list.append(qfCreateBuildMethod('YC_OVERNIGHT_INDEX_ELEMENT', {
    'TARGET': 'USD-Federal Funds-H.15-1B',
    'REFERENCE': 'SOFR-1B',
    'INSTANTANEOUS FORWARD RATE': 'USD-OIS-1B-IFR',
}))

bm_list.append(qfCreateBuildMethod('YC_IBOR_ELEMENT', {
    'TARGET': 'USD-LIBOR-BBA-3M',
    'REFERENCE': 'SOFR-1B',
    'INSTANTANEOUS FORWARD RATE': 'USD-LIBOR-BBA-3M-IFR',
}))

bm_list.append(qfCreateBuildMethod('YC_FUNDING_ELEMENT', {
    'TARGET': 'SOFR-1B-FLAT',
    'REFERENCE': 'SOFR-1B',
    'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-FLAT-IFR',
}))

bm_list.append(qfCreateBuildMethod('YC_FUNDING_ELEMENT', {
    'TARGET': 'USD-Federal Funds-H.15-1B-FLAT',
    'REFERENCE': 'USD-Federal Funds-H.15-1B',
    'INSTANTANEOUS FORWARD RATE': 'USD-OIS-1B-FLAT-IFR',
}))

bm_list.append(qfCreateBuildMethod('YC_COMMON', {
    'TARGET': 'USD',
    'FUNDING PARAMETERS': 'SOFR-1B-FLAT',
    'SOLVER METHOD': 'BRENT',
}))

build_method_collection = qfCreateModelBuildMethodCollection(bm_list)
qfDisplayModelBuildMethodCollection(build_method_collection)

,Name,Value
0,YC_OVERNIGHT_INDEX_ELEMENT,SOFR-1B
1,YC_OVERNIGHT_INDEX_ELEMENT,USD-FEDERAL FUNDS-H.15-1B
2,YC_IBOR_ELEMENT,USD-LIBOR-BBA-3M
3,YC_FUNDING_ELEMENT,SOFR-1B-FLAT
4,YC_FUNDING_ELEMENT,USD-FEDERAL FUNDS-H.15-1B-FLAT
5,YC_COMMON,USD


### Create Data Collection (USD)

Synthetic instantaneous forward rate curves. The `-FLAT` curves are flat zero spreads (IFR = 0), matching how `SOFR-1B-FLAT` / `USD-Federal Funds-H.15-1B-FLAT` are used elsewhere as the unspread CSA/funding overlay.

In [3]:
data_type = 'INSTANTANEOUS FORWARD RATE'
tenors = ['3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']
data_list = []

df = pd.DataFrame(index=tenors)
df['values'] = [0.0430, 0.0420, 0.0400, 0.0380, 0.0360, 0.0375, 0.0400]
data_list.append(qfCreateData1D(data_type, 'USD-SOFR-OIS-1B-IFR', df))

df = pd.DataFrame(index=tenors)
df['values'] = [0.0433, 0.0423, 0.0403, 0.0383, 0.0363, 0.0378, 0.0403]
data_list.append(qfCreateData1D(data_type, 'USD-OIS-1B-IFR', df))

df = pd.DataFrame(index=tenors)
df['values'] = [0.0455, 0.0445, 0.0420, 0.0395, 0.0370, 0.0385, 0.0405]
data_list.append(qfCreateData1D(data_type, 'USD-LIBOR-BBA-3M-IFR', df))

df = pd.DataFrame(index=['1Y', '10Y', '30Y'])
df['values'] = [0.0, 0.0, 0.0]
data_list.append(qfCreateData1D(data_type, 'USD-SOFR-OIS-1B-FLAT-IFR', df))

df = pd.DataFrame(index=['1Y', '10Y', '30Y'])
df['values'] = [0.0, 0.0, 0.0]
data_list.append(qfCreateData1D(data_type, 'USD-OIS-1B-FLAT-IFR', df))

data_collection = qfCreateDataCollection(data_list)
data_collection.display()

,Data Shape,Data Type,Data Convention
0,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-IFR
1,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-OIS-1B-IFR
2,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-LIBOR-BBA-3M-IFR
3,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-FLAT-IFR
4,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-OIS-1B-FLAT-IFR


### Create Model Yield Curve (USD)

In [4]:
value_date = '2026-06-25'
yc_usd = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)
qfDisplayModelValueDate(yc_usd), qfDisplayModelType(yc_usd)

('2026-06-25', 'YIELD_CURVE')

### Inspect Resulting Model

In [5]:
display(qfGetBuildMethodCollection(yc_usd).display())
display(qfGetDataCollectionFromModel(yc_usd).display())
print(f'USD yield curve model built successfully with {yc_usd.num_components} components.')

,Name,Value
0,YC_OVERNIGHT_INDEX_ELEMENT,SOFR-1B
1,YC_OVERNIGHT_INDEX_ELEMENT,USD-FEDERAL FUNDS-H.15-1B
2,YC_IBOR_ELEMENT,USD-LIBOR-BBA-3M
3,YC_FUNDING_ELEMENT,SOFR-1B-FLAT
4,YC_FUNDING_ELEMENT,USD-FEDERAL FUNDS-H.15-1B-FLAT
5,YC_COMMON,USD


,Data Shape,Data Type,Data Convention
0,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-IFR
1,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-OIS-1B-IFR
2,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-LIBOR-BBA-3M-IFR
3,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-FLAT-IFR
4,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-OIS-1B-FLAT-IFR


USD yield curve model built successfully with 5 components.


### Sanity Check: Discount Factors

In [6]:
expiry_date = '2027-06-25'
print('SOFR-1B           DF:', qfDiscountFactor(yc_usd, 'SOFR-1B', expiry_date))
print('USD-Federal Funds-H.15-1B             DF:', qfDiscountFactor(yc_usd, 'USD-Federal Funds-H.15-1B', expiry_date))
print('USD-LIBOR-BBA-3M  DF:', qfDiscountFactor(yc_usd, 'USD-LIBOR-BBA-3M', expiry_date))

SOFR-1B           DF: 0.959568828034545
USD-Federal Funds-H.15-1B             DF: 0.9204961454654816
USD-LIBOR-BBA-3M  DF: 0.9186922297745306


#### Reference-Index Resolution: `USD-Federal Funds-H.15-1B-FLAT` -> `USD-Federal Funds-H.15-1B`

`USD-Federal Funds-H.15-1B-FLAT` is a `FundingIdentifier` whose `reference_index` is `USD-Federal Funds-H.15-1B`. `USD-Federal Funds-H.15-1B` itself is a plain `ql.Index`
(not a `FundingIdentifier`), so the chain is two levels deep: `DF(USD-Federal Funds-H.15-1B-FLAT)` should equal `USD-Federal Funds-H.15-1B`'s own discount
factor times `USD-Federal Funds-H.15-1B-FLAT`'s own discount factor.

In [7]:
ff_own = yc_usd.retrieve_model_component('USD-Federal Funds-H.15-1B').discount_factor(Date(expiry_date))
ff_flat_own = yc_usd.retrieve_model_component('USD-Federal Funds-H.15-1B-FLAT').discount_factor(Date(expiry_date))
sofr_own = yc_usd.retrieve_model_component('SOFR-1B').discount_factor(Date(expiry_date))

expected_ff_1b_flat_df = ff_own * ff_flat_own *sofr_own
actual_ff_1b_flat_df = qfDiscountFactor(yc_usd, 'USD-Federal Funds-H.15-1B-FLAT', expiry_date)

print('own DF  USD-Federal Funds-H.15-1B       :', ff_own)
print('own DF  USD-Federal Funds-H.15-1B-FLAT  :', ff_flat_own)
print('own DF  SOFR-1B     :', sofr_own)

print('expected DF(USD-Federal Funds-H.15-1B-FLAT) = USD-Federal Funds-H.15-1B(own) * USD-Federal Funds-H.15-1B-FLAT(own):', expected_ff_1b_flat_df)
print('actual   DF(USD-Federal Funds-H.15-1B-FLAT) via qfDiscountFactor          :', actual_ff_1b_flat_df)
assert np.isclose(expected_ff_1b_flat_df, actual_ff_1b_flat_df), 'reference-index resolution is broken'
print('PASS: reference chain resolved correctly')

own DF  USD-Federal Funds-H.15-1B       : tensor(0.9593, dtype=torch.float64)
own DF  USD-Federal Funds-H.15-1B-FLAT  : tensor(1., dtype=torch.float64)
own DF  SOFR-1B     : tensor(0.9596, dtype=torch.float64)
expected DF(USD-Federal Funds-H.15-1B-FLAT) = USD-Federal Funds-H.15-1B(own) * USD-Federal Funds-H.15-1B-FLAT(own): tensor(0.9205, dtype=torch.float64)
actual   DF(USD-Federal Funds-H.15-1B-FLAT) via qfDiscountFactor          : 0.9204961454654816
PASS: reference chain resolved correctly


### Gradient of Discount Factor w.r.t. Each Component's State Data

`qfDiscountFactor(..., calc_grad=True)` now forwards `calc_grad` all the way down to `Interpolator1D.integrate`, so the discount factor itself stays a differentiable torch tensor -- no need to reimplement the integral/exp logic by hand.

In [8]:
def discount_factor_gradient_wrt_state(model, target, expiry_date):
    component = model.retrieve_model_component(target)
    component.state_data_interpolator.values_.grad = None
    df = qfDiscountFactor(model, target, expiry_date, calc_grad=True)
    df.backward()
    return df.detach(), component.state_data_interpolator.values_.grad.numpy()

for target in ['SOFR-1B', 'USD-Federal Funds-H.15-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT', 'USD-Federal Funds-H.15-1B-FLAT']:
    df, grad = discount_factor_gradient_wrt_state(yc_usd, target, expiry_date)
    print(f'{target:18s} DF={float(df):.6f}  dDF/d(IFR)={grad}')

SOFR-1B            DF=0.959569  dDF/d(IFR)=[-0.24186392 -0.24712184 -0.47058307  0.          0.          0.
 -0.        ]
USD-Federal Funds-H.15-1B DF=0.920496  dDF/d(IFR)=[-0.23201547 -0.23705928 -0.4514214   0.          0.          0.
 -0.        ]
USD-LIBOR-BBA-3M   DF=0.918692  dDF/d(IFR)=[-0.23156078 -0.23911168 -0.44801977  0.          0.          0.
 -0.        ]
SOFR-1B-FLAT       DF=0.959569  dDF/d(IFR)=[-0.95956883  0.         -0.        ]
USD-Federal Funds-H.15-1B-FLAT DF=0.920496  dDF/d(IFR)=[-0.92049615  0.         -0.        ]


### `YieldCurve.get_gradient` -- Harvesting Gradients Across All Components

`get_gradient(reset=False)` does not call `.backward()` itself -- it only harvests whatever `.grad` torch has already accumulated on each component's `Interpolator1D.values_` tensor, concatenated **in dependency order** (a component's reference curve always comes before the component itself, via `_gradient_component_order`), and records the per-component lengths in `gradient_lengths_` / `gradient_component_order_` so the flat vector can be sliced back apart.

`reset=True` clears every harvested `.grad` afterwards, which is what a valuation engine should use between pricing different products on the same model -- otherwise torch silently *accumulates* gradients across `.backward()` calls instead of overwriting them.

In [9]:
print('dependency-respecting component order:', yc_usd._gradient_component_order())

# baseline: nothing has called .backward() yet on a *fresh* set of grads
for target in ['SOFR-1B', 'USD-Federal Funds-H.15-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT', 'USD-Federal Funds-H.15-1B-FLAT']:
    component = yc_usd.retrieve_model_component(target)
    component.state_data_interpolator.values_.grad = None

grad_before = yc_usd.get_gradient(reset=False)
print('gradient before any backward() (should be all zero):', grad_before)

dependency-respecting component order: ['SOFR-1B', 'USD-FEDERAL FUNDS-H.15-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT', 'USD-FEDERAL FUNDS-H.15-1B-FLAT']
gradient before any backward() (should be all zero): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0.]


In [10]:
# price USD-Federal Funds-H.15-1B-FLAT with calc_grad=True and backward() once
pv = qfDiscountFactor(yc_usd, 'USD-Federal Funds-H.15-1B-FLAT', expiry_date, calc_grad=True)
pv.backward()

grad_after_one_backward = yc_usd.get_gradient(reset=False)
print('PV (DF of USD-Federal Funds-H.15-1B-FLAT):', float(pv.detach()))
print('component_order_         :', yc_usd.component_order_)
print('gradient_lengths_        :', yc_usd.gradient_lengths_)
print('concatenated gradient     :', grad_after_one_backward)

# slice the flat vector back into per-component pieces using the recorded lengths/order
offset = 0
for key, length in zip(yc_usd.component_order_, yc_usd.gradient_lengths_):
    print(f'{key:18s}', grad_after_one_backward[offset:offset + length])
    offset += length

PV (DF of USD-Federal Funds-H.15-1B-FLAT): 0.9204961454654816
component_order_         : ['SOFR-1B', 'USD-FEDERAL FUNDS-H.15-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT', 'USD-FEDERAL FUNDS-H.15-1B-FLAT']
gradient_lengths_        : [7, 7, 7, 3, 3]
concatenated gradient     : [-0.23201547 -0.23705928 -0.4514214   0.          0.          0.
 -0.         -0.23201547 -0.23705928 -0.4514214   0.          0.
  0.         -0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
 -0.92049615  0.         -0.        ]
SOFR-1B            [-0.23201547 -0.23705928 -0.4514214   0.          0.          0.
 -0.        ]
USD-FEDERAL FUNDS-H.15-1B [-0.23201547 -0.23705928 -0.4514214   0.          0.          0.
 -0.        ]
USD-LIBOR-BBA-3M   [0. 0. 0. 0. 0. 0. 0.]
SOFR-1B-FLAT       [0. 0. 0.]
USD-FEDERAL FUNDS-H.15-1B-FLAT [-0.92049615  0.         -0.        ]


In [11]:
# backward() a second time WITHOUT resetting in between -> torch accumulates, gradient should double
pv2 = qfDiscountFactor(yc_usd, 'USD-Federal Funds-H.15-1B-FLAT', expiry_date, calc_grad=True)
pv2.backward()

grad_accumulated = yc_usd.get_gradient(reset=False)
ratio = np.divide(grad_accumulated, grad_after_one_backward, out=np.zeros_like(grad_accumulated), where=grad_after_one_backward != 0)
print('gradient after 2nd backward, no reset in between:', grad_accumulated)
print('ratio vs. single backward (should be ~2 on nonzero entries):', ratio)

gradient after 2nd backward, no reset in between: [-0.46403093 -0.47411856 -0.90284279  0.          0.          0.
 -0.         -0.46403093 -0.47411856 -0.90284279  0.          0.
  0.         -0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
 -1.84099229  0.         -0.        ]
ratio vs. single backward (should be ~2 on nonzero entries): [2. 2. 2. 0. 0. 0. 0. 2. 2. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 2. 0. 0.]


In [12]:
# get_gradient(reset=True) should clear every grad it just harvested,
# so the very next valuation does not pick up stale gradient from this one.
_ = yc_usd.get_gradient(reset=True)
grad_after_reset = yc_usd.get_gradient(reset=False)
print('gradient immediately after reset=True (should be all zero):', grad_after_reset)

gradient immediately after reset=True (should be all zero): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0.]
